In [6]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import re


In [7]:
USER_AGENT = {"User-Agent": "Mozilla/5.0"}
REQUEST_TIMEOUT = 15
SCRAPE_DELAY = 1

RAW_OUTPUT_FILE = "military raw data.csv"
FINAL_OUTPUT_FILE = "military pivot data.csv"
LINKS_FILE = "links for military data.txt"

In [8]:
def extract_valid_url(text_line):
    """Extracts a valid URL from a given line."""
    match = re.search(r"(https?://[^\s']+)", text_line)
    return match.group(1) if match else None



In [9]:
def clean_numeric_value(value):
    """Converts metric value to float-safe format."""
    return (
        str(value)
        .replace(",", "")
        .strip()
    )



In [10]:
all_data = []
processed_pages = 0
skipped_pages = 0

with open(LINKS_FILE, "r") as file:
    raw_lines = [line.strip() for line in file if line.strip()]

metric_urls = []
for line in raw_lines:
    url = extract_valid_url(line)
    if url:
        metric_urls.append(url)

print(f"Metric links detected: {len(metric_urls)}")

Metric links detected: 55


In [11]:
for url in metric_urls:

    if "countries-listing.php" in url:
        skipped_pages += 1
        continue

    print("Fetching:", url)

    try:
        response = requests.get(
            url,
            headers=USER_AGENT,
            timeout=REQUEST_TIMEOUT
        )

        if response.status_code != 200:
            print("Skipped (bad response):", url)
            continue

        soup = BeautifulSoup(response.text, "html.parser")
        rows = soup.find_all("div", class_="topRow")

        if not rows:
            print("No usable rows:", url)
            continue

        metric_name = url.split("/")[-1].replace(".php", "")

        for row in rows:
            span_values = [s.get_text(strip=True) for s in row.find_all("span")]

            if len(span_values) < 3:
                continue

            country = None
            value = None

            for text in span_values:
                if not country and any(ch.isalpha() for ch in text):
                    country = text
                elif country and any(ch.isdigit() for ch in text):
                    value = text
                    break

            if country and value:
                all_data.append({
                    "country_name": country,
                    "metric": metric_name,
                    "value": value
                })

        processed_pages += 1
        time.sleep(SCRAPE_DELAY)

    except Exception as err:
        print("Error occurred:", err)

print("Scraping finished")
print("Pages processed:", processed_pages)
print("Pages skipped:", skipped_pages)
print("Total records:", len(all_data))

Fetching: https://www.globalfirepower.com/total-population-by-country.php
Fetching: https://www.globalfirepower.com/available-military-manpower.php
Error occurred: ('Connection aborted.', ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None))
Fetching: https://www.globalfirepower.com/manpower-fit-for-military-service.php
Fetching: https://www.globalfirepower.com/manpower-reaching-military-age-annually.php
Fetching: https://www.globalfirepower.com/active-military-manpower.php
Error occurred: ('Connection aborted.', ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None))
Fetching: https://www.globalfirepower.com/active-reserve-military-manpower.php
Fetching: https://www.globalfirepower.com/manpower-paramilitary.php
Fetching: https://www.globalfirepower.com/aircraft-total.php
Fetching: https://www.globalfirepower.com/aircraft-total-fighters.php
Fetching: https://www.globalfir

In [12]:
raw_df = pd.DataFrame(all_data)
raw_df.to_csv(RAW_OUTPUT_FILE, index=False)
print(f"Saved raw file: {RAW_OUTPUT_FILE}")


Saved raw file: military raw data.csv


In [13]:
pivot_df = raw_df.pivot_table(
    index="country_name",
    columns="metric",
    values="value",
    aggfunc="first"
).reset_index()

In [14]:
rank_df = pivot_df.copy()

for col in rank_df.columns:
    if col == "country_name":
        continue

    rank_df[col] = (
        rank_df[col]
        .astype(str)
        .apply(clean_numeric_value)
        .str.extract(r"([\d\.]+)")
        .astype(float)
        .fillna(0)
    )

metric_cols = [c for c in rank_df.columns if c != "country_name"]

for col in metric_cols:
    max_val = rank_df[col].max()
    rank_df[col] = rank_df[col] / max_val if max_val > 0 else 0

rank_df["global_rank"] = (
    rank_df[metric_cols]
    .sum(axis=1)
    .rank(ascending=False, method="dense")
    .astype(int)
)

In [15]:
final_df = pivot_df.merge(
    rank_df[["country_name", "global_rank"]],
    on="country_name",
    how="left"
)

final_df = final_df[
    ["global_rank", "country_name"] +
    [c for c in final_df.columns if c not in ["global_rank", "country_name"]]
]

final_df = final_df.sort_values("global_rank")
final_df.to_csv(FINAL_OUTPUT_FILE, index=False)

print(f"Final ranked file saved: {FINAL_OUTPUT_FILE}")

Final ranked file saved: military pivot data.csv
